In [1]:
import numpy as np
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
from sklearn.model_selection import train_test_split, GridSearchCV, GroupKFold
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectKBest, f_regression
import pandas as pd
import random

SEED_LIST = [10, 20, 50, 100, 150, 200, 512, 1024, 2048, 8192]
K_VALUES = [2, 3, 4, 5, 6, 7, 8]
GAP_THRESHOLD = 0.15
CV_FOLDS = 5

param_space = {
    'n_estimators': [50, 100, 200, 300],
    'max_depth': [None, 10, 20, 30],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'max_features': ['sqrt', 'log2', None]
}

baseline_param_space = {
    'n_estimators': [50, 80, 100],
    'max_depth': [3, 5, 7],
    'min_samples_split': [10, 15, 20],
    'min_samples_leaf': [5, 8, 10],
    'max_features': ['sqrt', 'log2']
}


def build_reg_model(params, seed):
    return RandomForestRegressor(
        n_estimators=params['n_estimators'],
        max_depth=params['max_depth'],
        min_samples_split        min_samples_split=paramsparams['min_samples_split'],
        min_samples_leaf        min_samples_leaf=paramsparams['min_samples_leaf'],
        max_features        max_features=paramsparams['max_features'],
        bootstrap        bootstrap=True,
        random_state        random_state=seedseed,
        n_jobs        n_jobs=-1
    )


def mean_relative_error(y_truey_true, y_pred y_pred)::
    mask     mask = y_true  y_true != 0
    if np np.sum(maskmask) == 0::
        return 0.0
    return np np.mean(npnp.abs((y_truey_true[maskmask] - y_pred y_pred[maskmask]) / y_true y_true[maskmask]))


def calculate_metrics(y_truey_true, y_pred y_pred)::
    mae     mae = mean_absolute_error mean_absolute_error(y_truey_true, y_pred y_pred)
    mse     mse = mean_squared_error mean_squared_error(y_truey_true, y_pred y_pred)
    rmse     rmse = np np.sqrt(msemse)
    mre     mre = mean_relative_error mean_relative_error(y_truey_true, y_pred y_pred)
    r2     r2 = r2_score r2_score(y_truey_true, y_pred y_pred)
    return {'mae': mae: mae, 'mse': mse: mse, 'rmse': rmse: rmse, 'mre': mre: mre, 'r2': r2: r2}


print("=" * 60)
print("Step 0: Data Loading & Basic Preprocessing")
print("-" * 60)

pattern pattern = pd pd.read_csv('4-pattern2.csv', header header=None, encoding encoding='utf-8-sig').values.astype('float64')
pattern pattern = np np.where(npnp.isinf(patternpattern), np np.nan, pattern pattern)
mean_val mean_val = np np.nanmean(patternpattern) if not np np.isnan(npnp.nanmean(patternpattern)) else 0
pattern pattern = np np.nan_to_num(patternpattern, nan nan=mean_valmean_val)

min_vals min_vals = np np.min(patternpattern, axis axis=0)
max_vals = np.max(pattern, axis=0)
range_vals = np.where(max_vals - min_vals == 0, 1, max_vals - min_vals)
pattern_normalized = (pattern - min_vals) / range_vals

scaler_spectral = StandardScaler()
pattern_scaled = scaler_spectral.fit_transform(pattern_normalized)

print("\nLoading image features...")
image_features = pd.read_csv('image_features_mobilenet_large.csv', header=None).values.astype('float64')
scaler_image = StandardScaler()
image_features_scaled = scaler_image.fit_transform(image_features)

pca_image = PCA(n_components=0.95)
pattern_image_pca_full = pca_image.fit_transform(image_features_scaled)
image_pca_variance = pca_image.explained_variance_ratio_
print(f"Image PCA completed: {pca_image.n_components_} components retained, cumulative variance ratio: {np.sum(image_pca_variance):.4f}")
print(f"Top 5 image PC variance ratios: {[round(v, 4) for v in image_pca_variance[:5]]}")

label_data = pd.read_csv('label_rIMS2-2.csv', header=None, encoding='utf-8-sig').values.astype('float64')
groups = label_data[:, 0]
label_c = label_data[:, 1]
label_c = np.exp(label_c)
total_samples = len(label_c)
unique_groups = np.unique(groups)
test_size_groups = int(0.2 * len(unique_groups))

print("\nData loading & preprocessing completed.\n")


def run_single_seed(seed):
    np.random.seed(seed)
    random.seed(seed)

    train_groups, test_groups = train_test_split(
        unique_groups,
        test_size=test_size_groups,
        random_state=seed
    )

    train_mask = np.isin(groups, train_groups)
    test_mask = np.isin(groups, test_groups)
    train_indices = np.where(train_mask)[0]
    test_indices = np.where(test_mask)[0]

    X_train_raw_spectral = pattern_scaled[train_mask]
    X_test_raw_spectral = pattern_scaled[test_mask]
    y_train = label_c[train_mask]
    y_test = label_c[test_mask]

    pca_spectral = PCA(n_components=6)
    X_train_spectral_pca = pca_spectral.fit_transform(X_train_raw_spectral)
    X_test_spectral_pca = pca_spectral.transform(X_test_raw_spectral)

    X_train_image_pca_full = pattern_image_pca_full[train_mask]
    X_test_image_pca_full = pattern_image_pca_full[test_mask]

    train_groups_cv = groups[train_mask]
    group_kfold = GroupKFold(n_splits=CV_FOLDS)

    base_model = RandomForestRegressor(bootstrap=True, random_state=seed, n_jobs=-1)
    grid_search_baseline = GridSearchCV(
        estimator=base_model,
        param_grid=baseline_param_space,
        cv=group_kfold,
        scoring='r2',
        n_jobs=-1,
        verbose=0
    )
    grid_search_baseline.fit(X_train_spectral_pca, y_train, groups=train_groups_cv)
    baseline_best_params = grid_search_baseline.best_params_

    X_tr_base, X_val_base, y_tr_base, y_val_base, tr_idx_base, val_idx_base = train_test_split(
        X_train_spectral_pca, y_train, train_indices,
        test_size=0.2,
        random_state=seed
    )

    baseline_model = build_reg_model(baseline_best_params, seed)
    baseline_model.fit(X_tr_base, y_tr_base)

    y_tr_pred_base = baseline_model.predict(X_tr_base)
    y_val_pred_base = baseline_model.predict(X_val_base)
    y_train_pred_base = baseline_model.predict(X_train_spectral_pca)
    y_test_pred_base = baseline_model.predict(X_test_spectral_pca)

    tr_metrics_base = calculate_metrics(y_tr_base, y_tr_pred_base)
    val_metrics_base = calculate_metrics(y_val_base, y_val_pred_base)
    train_metrics_base = calculate_metrics(y_train, y_train_pred_base)
    test_metrics_base = calculate_metrics(y_test, y_test_pred_base)

    baseline_result = {
        'Seed': seed,
        'Train_R2': round(train_metrics_base['r2'], 6),
        'Train_MAE': round(train_metrics_base['mae'], 6),
        'Train_MSE': round(train_metrics_base['mse'], 6),
        'Train_MRE': round(train_metrics_base['mre'], 6),
        'Train_RMSE': round(train_metrics_base['rmse'], 6),
        'Val_R2': round(val_metrics_base['r2'], 6),
        'Val_MAE': round(val_metrics_base['mae'], 6),
        'Val_MSE': round(val_metrics_base['mse'], 6),
        'Val_MRE': round(val_metrics_base['mre'], 6),
        'Val_RMSE': round(val_metrics_base['rmse'], 6),
        'Test_R2': round(test_metrics_base['r2'], 6),
        'Test_MAE': round(test_metrics_base['mae'], 6),
        'Test_MSE': round(test_metrics_base['mse'], 6),
        'Test_MRE': round(test_metrics_base['mre'], 6),
        'Test_RMSE': round(test_metrics_base['rmse'], 6),
        'Best_Params': str(baseline_best_params)
    }

    all_k_results = []
    best_overall = None

    for k in K_VALUES:
        selector = SelectKBest(score_func=f_regression, k=k)
        X_train_image_pca_selected = selector.fit_transform(X_train_image_pca_full, y_train)
        X_test_image_pca_selected = selector.transform(X_test_image_pca_full)
        selected_indices = selector.get_support(indices=True)
        selected_variance = image_pca_variance[selected_indices]
        cumulative_variance = np.sum(selected_variance)

        X_train_fused = np.hstack([X_train_spectral_pca, X_train_image_pca_selected])
        X_test_fused = np.hstack([X_test_spectral_pca, X_test_image_pca_selected])

        base_model = RandomForestRegressor(bootstrap=True, random_state=seed, n_jobs=-1)
        grid_search = GridSearchCV(
            estimator=base_model,
            param_grid=param_space,
            cv=group_kfold,
            scoring='r2',
            n_jobs=-1,
            verbose=0
        )
        grid_search.fit(X_train_fused, y_train, groups=train_groups_cv)
        best_params = grid_search.best_params_
        best_cv_score = grid_search.best_score_

        X_tr, X_val, y_tr, y_val, tr_idx, val_idx = train_test_split(
            X_train_fused, y_train, train_indices,
            test_size=0.2,
            random_state=seed
        )

        final_model = build_reg_model(best_params, seed)
        final_model.fit(X_tr, y_tr)

        y_tr_pred = final_model.predict(X_tr)
        y_val_pred = final_model.predict(X_val)
        y_train_pred = final_model.predict(X_train_fused)
        y_test_pred = final_model.predict(X_test_fused)

        tr_metrics = calculate_metrics(y_tr, y_tr_pred)
        val_metrics = calculate_metrics(y_val, y_val_pred)
        train_metrics = calculate_metrics(y_train, y_train_pred)
        test_metrics = calculate_metrics(y_test, y_test_pred)

        result_entry = {
            'Seed': seed,
            'K': k,
            'Selected_Image_PCs': str(selected_indices),
            'Cumulative_Variance_Ratio': round(cumulative_variance, 4),
            'Best_Params': str(best_params),
            'CV_R2': round(best_cv_score, 6),
            'Train_R2': round(train_metrics['r2'], 6),
            'Train_MAE': round(train_metrics['mae'], 6),
            'Train_MSE': round(train_metrics['mse'], 6),
            'Train_MRE': round(train_metrics['mre'], 6),
            'Train_RMSE': round(train_metrics['rmse'], 6),
            'Val_R2': round(val_metrics['r2'], 6),
            'Val_MAE': round(val_metrics['mae'], 6),
            'Val_MSE': round(val_metrics['mse'], 6),
            'Val_MRE': round(val_metrics['mre'], 6),
            'Val_RMSE': round(val_metrics['rmse'], 6),
            'Test_R2': round(test_metrics['r2'], 6),
            'Test_MAE': round(test_metrics['mae'], 6),
            'Test_MSE': round(test_metrics['mse'], 6),
            'Test_MRE': round(test_metrics['mre'], 6),
            'Test_RMSE': round(test_metrics['rmse'], 6)
        }
        all_k_results.append(result_entry)

        if best_overall is None or test_metrics['r2'] > best_overall['Test_R2']:
            best_overall = result_entry

    return baseline_result, all_k_results, best_overall


print("=" * 60)
print("Step 1: Random Seed Robustness Test Running")
print(f"Total seeds to test: {len(SEED_LIST)}")
print(f"Seed list: {SEED_LIST}")
print("-" * 60)

all_baseline_results = []
all_k_total_results = []
all_best_per_seed = []

for idx, seed in enumerate(SEED_LIST, 1):
    print(f"\n[{idx}/{len(SEED_LIST)}] Testing seed = {seed} ...")
    baseline_res, k_res_list, best_res = run_single_seed(seed)

    all_baseline_results.append(baseline_res)
    all_k_total_results.extend(k_res_list)
    all_best_per_seed.append(best_res)

    print(f"  Completed | Best Test R²: {best_res['Test_R2']:.6f} | Val R²: {best_res['Val_R2']:.6f}")


print("\n" + "=" * 60)
print("Step 2: Robustness Test Summary & Statistics")
print("-" * 60)

baseline_df = pd.DataFrame(all_baseline_results)
all_k_df = pd.DataFrame(all_k_total_results)
best_per_seed_df = pd.DataFrame(all_best_per_seed)

stats_columns = ['Test_R2', 'Test_MAE', 'Test_MRE', 'Test_RMSE']
fused_stats = pd.DataFrame({
    'Metric': stats_columns,
    'Mean': [best_per_seed_df[col].mean() for col in stats_columns],
    'Std': [best_per_seed_df[col].std() for col in stats_columns],
    'Min': [best_per_seed_df[col].min() for col in stats_columns],
    'Max': [best_per_seed_df[col].max() for col in stats_columns],
    'Coefficient_of_Variation': [
        best_per_seed_df[col].std() / abs(best_per_seed_df[col].mean())
        if best_per_seed_df[col].mean() != 0 else 0
        for col in stats_columns
    ]
})

print("\nFused Model - Test Set Robustness Statistics (Best Result per Seed):")
print(fused_stats.round(6).to_string(index=False))

print("\n\nDetailed Best Results per Seed (Fused Model):")
display_cols = ['Seed', 'CV_R2', 'Val_R2', 'Val_MAE', 'Val_RMSE', 'Test_R2', 'Test_MAE', 'Test_RMSE']
print(best_per_seed_df[display_cols].to_string(index=False))

print("\n\nBaseline Model - Test Set Robustness Statistics:")
baseline_stats = pd.DataFrame({
    'Metric': stats_columns,
    'Mean': [baseline_df[col].mean() for col in stats_columns],
    'Std': [baseline_df[col].std() for col in stats_columns],
    'Min': [baseline_df[col].min() for col in stats_columns],
    'Max': [baseline_df[col].max() for col in stats_columns]
})
print(baseline_stats.round(6).to_string(index=False))

print("\n\nDetailed Baseline Results per Seed:")
baseline_display_cols = ['Seed', 'Val_R2', 'Val_MAE', 'Val_RMSE', 'Test_R2', 'Test_MAE', 'Test_RMSE']
print(baseline_df[baseline_display_cols].to_string(index=False))


output_file = 'Random_Seed_Robustness_Test_Results_IMS.xlsx'
with pd.ExcelWriter(output_file, engine='openpyxl') as writer:
    best_per_seed_df.to_excel(writer, sheet_name='Best_Fused_Model_Per_Seed', index=False)
    all_k_df.to_excel(writer, sheet_name='All_K_Results_All_Seeds', index=False)
    baseline_df.to_excel(writer, sheet_name='Baseline_Results_Per_Seed', index=False)
    fused_stats.to_excel(writer, sheet_name='Fused_Model_Robustness_Stats', index=False)
    baseline_stats.to_excel(writer, sheet_name='Baseline_Model_Robustness_Stats', index=False)

print(f"\nAll results saved to file: {output_file}")
print("=" * 60)

Step 0: Data Loading & Basic Preprocessing
------------------------------------------------------------

Loading image features...
Image PCA completed: 186 components retained, cumulative variance ratio: 0.9503
Top 5 image PC variance ratios: [np.float64(0.1658), np.float64(0.0762), np.float64(0.0614), np.float64(0.0468), np.float64(0.0415)]

Data loading & preprocessing completed.

Step 1: Random Seed Robustness Test Running
Total seeds to test: 10
Seed list: [10, 20, 50, 100, 150, 200, 512, 1024, 2048, 8192]
------------------------------------------------------------

[1/10] Testing seed = 10 ...
  Completed | Best Test R²: 0.758511 | Val R²: 0.835632

[2/10] Testing seed = 20 ...
  Completed | Best Test R²: 0.845850 | Val R²: 0.871027

[3/10] Testing seed = 50 ...
  Completed | Best Test R²: 0.798842 | Val R²: 0.815550

[4/10] Testing seed = 100 ...
  Completed | Best Test R²: 0.880100 | Val R²: 0.783155

[5/10] Testing seed = 150 ...
  Completed | Best Test R²: 0.844338 | Val R²: 